1. Shape of embedding [n_t]
2. Shape of output[n_vocab]

Build the network: x_n -> f(x_n) -> error -> x_n-1 -> 
For each example, run the inference pass, then run the backward learning pass 

For inference: 
1. Get the input
2. Initialise the layers to a random value 
3. Pad the weights & the layers [do this beforehand]
4. Calculate the errors, pre-activation for all layers
5. Run the gradient step on the layers

TODO: 
1. Can PC learning be applied to transformers -> learning attention is the issue, "attention-like patterns", where the entire text is processed via linear layers
2. Building an RNN network trained for auto-regressive text generation using PCN

In [ ]:
import jax.numpy as jnp
from jax.nn import relu 
import jax

class TextTokeniser: 
    def tokenize(txt): 
        return []

class PCN: 
    num_layers = 0
    ## These are stored as list of jnp arrays, 
    # however will be padded & converted to jnp arrays 
    layers = []
    weights = []
    eta_inference = 0.1
    eta_learn = 0.1
    t_infer = 50

    def __init__(self,layers,weights): 
        self.layers = layers
        self.weights = weights
        self.num_layers = len(layers)

## Use a data-base of text; just what comes after in the alphabet 
## Use rev mode grad diff & update on that basis.

##Pads the layer & weights tensors given 
# the sizes may differ across layers: 
# Input is t which is a list of jnp Arrays of shape (nd_i) 
# or shape (nd_i,nd_i-1)
def build_padded_tensor(t,max_size,dim): 
    return []

# layers shape: [nl,nd_i]
# weights shape: [n_l-1,nd_i-1,nd_i]
## layers proceed from x_l.... x_0
## layers proceed from W_l-1 .... W_0
## Errors are from e_l-1....e_0
@jax.jit
def errorsnpreactivations(layers: jnp.array, weights: jnp.array):
    # indices = [range(5)]
    first_layer_index = layers.shape[1] - 1
    upper_layers = layers[0:first_layer_index-1,:]
    # upper layers shape: [n_l-1,nd]
    # weights shape: [n_l-1,nd,nd]
    # preactivation shape: [n_l-1,nd]
    preactivation = weights @ upper_layers
    activation = relu(preactivation)
    lower_layers = layers[1:first_layer_index,:]

    # activation = relu(layers[index] @ weights[index])
    # errors shape: [n_l-1,nd]
    errors = activation - lower_layers
    return errors, preactivation

@jax.jit
def mse(errors): 
    jnp.sum(jnp.square(errors))

##Layers shape: [nl,nd]; without padding [nl,nd_i]
## Errors shape: [nl-1,nd]; without padding [nl-1,nd_i-1]
## Preactivation shape: [nl-1,nd]; without padding [nl-1,nd_i-1]
## Weights shape: [nl-1, nd, nd]; without padding [n_l-1,nd_i-1,nd_i]
def update_layers(layers,weights,errors,preactivations,eta_inference):
    errors_shifted = jnp.pad(errors[0:len(errors)-1,:],((0,1),(0,0)),mode='empty')
    print("Errors shifted in update_layers",errors_shifted)
    first_layer_index = layers.shape[1] - 1
    upper_layers = layers[0:first_layer_index-1,:]
    drelu = jax.grad(jax.relu)
    grad = errors_shifted - jnp.transpose(weights) @ (errors * drelu(preactivations))
    upper_layers = upper_layers - eta_inference * grad
    # Once done return the layers: 
    return jnp.append(upper_layers,layers[len(layers)-1])

#Get initialised layers & weights
def update_layers(t,layers,weights,etas):
    for i in range(t): 
        errors, preactivation = errorsnpreactivations(layers,weights)
        layers = update_layers(layers,weights,errors,preactivation,etas[0])
    layers,weights


